<a href="https://colab.research.google.com/github/vedantyeole0207/Media-Bias-Prediction/blob/vedantyeole0207%2FUsed-Car-Price-Prediction-Full-Stack/Political_Media_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# 1) Restart Colab runtime first, then run this whole cell (exact order is important)
import os

# Ensure XLA is fully disabled before TF is imported
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
# Optional: also disable any JIT optimization
os.environ["TF_DISABLE_JIT"] = "1"

# Now import TF and keras_nlp
import tensorflow as tf
import keras_nlp
import pandas as pd
from sklearn.model_selection import train_test_split

In [17]:
print("TF version:", tf.__version__)
print("JIT enabled (should be False):", tf.config.optimizer.get_jit())
print("GPUs:", tf.config.list_physical_devices("GPU"))

TF version: 2.19.1
JIT enabled (should be False): 
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import keras_nlp

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
import re
import nltk
from nltk.corpus import stopwords
import numpy as np
from nltk.stem import WordNetLemmatizer
from sklearn.compose import ColumnTransformer
import tensorflow as tf

In [4]:
file_path = 'political_social_media.csv'
data = pd.read_csv(file_path, encoding='latin1')

In [5]:
data.head(5)

,_unit_id,_golden,_unit_state,_trusted_judgments,_last_judgment_at,audience,audience:confidence,bias,bias:confidence,message,...,orig__golden,audience_gold,bias_gold,bioid,embed,id,label,message_gold,source,text
0,766192484,False,finalized,1,8/4/15 21:17,national,1.0,partisan,1.0,policy,...,NaN,NaN,NaN,R000596,"<blockquote class=""twitter-tweet"" width=""450"">...",3.83249E+17,From: Trey Radel (Representative from Florida),NaN,twitter,RT @nowthisnews: Rep. Trey Radel (R- #FL) slam...
1,766192485,False,finalized,1,8/4/15 21:20,national,1.0,partisan,1.0,attack,...,NaN,NaN,NaN,M000355,"<blockquote class=""twitter-tweet"" width=""450"">...",3.11208E+17,From: Mitch McConnell (Senator from Kentucky),NaN,twitter,VIDEO - #Obamacare: Full of Higher Costs and ...
2,766192486,False,finalized,1,8/4/15 21:14,national,1.0,neutral,1.0,support,...,NaN,NaN,NaN,S001180,"<blockquote class=""twitter-tweet"" width=""450"">...",3.39069E+17,From: Kurt Schrader (Representative from Oregon),NaN,twitter,Please join me today in remembering our fallen...
3,766192487,False,finalized,1,8/4/15 21:08,national,1.0,neutral,1.0,policy,...,NaN,NaN,NaN,C000880,"<blockquote class=""twitter-tweet"" width=""450"">...",2.98528E+17,From: Michael Crapo (Senator from Idaho),NaN,twitter,RT @SenatorLeahy: 1st step toward Senate debat...
4,766192488,False,finalized,1,8/4/15 21:26,national,1.0,partisan,1.0,policy,...,NaN,NaN,NaN,U000038,"<blockquote class=""twitter-tweet"" width=""450"">...",4.07643E+17,From: Mark Udall (Senator from Colorado),NaN,twitter,.@amazon delivery #drones show need to update ...


In [6]:
print(type(data))
message_data = data['message']
bias_data = data['bias']
text_data = data['text']
id = 3
print(np.array(text_data)[id])

##Getting the classes of messages and bias
message_classes = message_data.unique()
bias_classes = bias_data.unique()
print(message_classes, bias_classes)

<class 'pandas.core.frame.DataFrame'>
RT @SenatorLeahy: 1st step toward Senate debate on Leahy-Crapo #VAWA bill is the SenateÛªs procedural vote today at 5:30 pm
['policy' 'attack' 'support' 'information' 'mobilization' 'personal'
 'other' 'constituency' 'media'] ['partisan' 'neutral']


In [7]:
data = data[['text', 'bias']].dropna()

# Inspect label distribution
print(data['bias'].value_counts())

bias
neutral     3689
partisan    1311
Name: count, dtype: int64


In [8]:
# Clean text a bit (remove URLs, mentions, hashtags)
import re
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    return text.strip()

data['clean_text'] = data['text'].apply(clean_text)
data.head()


,text,bias,clean_text
0,RT @nowthisnews: Rep. Trey Radel (R- #FL) slam...,partisan,RT : Rep. Trey Radel (R- FL) slams Obamacare. ...
1,VIDEO - #Obamacare: Full of Higher Costs and ...,partisan,VIDEO - Obamacare: Full of Higher Costs and B...
2,Please join me today in remembering our fallen...,neutral,Please join me today in remembering our fallen...
3,RT @SenatorLeahy: 1st step toward Senate debat...,neutral,RT : 1st step toward Senate debate on Leahy-Cr...
4,.@amazon delivery #drones show need to update ...,partisan,. delivery drones show need to update law to p...


In [35]:
data['label'] = data['bias'].map({'neutral': 0, 'partisan': 1})

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['label'], test_size=0.2, random_state=42, stratify=data['bias'])

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

model = LogisticRegression(max_iter=300)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.781
              precision    recall  f1-score   support

           0       0.78      0.98      0.87       738
           1       0.79      0.22      0.35       262

    accuracy                           0.78      1000
   macro avg       0.79      0.60      0.61      1000
weighted avg       0.78      0.78      0.73      1000



In [40]:
sample = ["The administration is destroying our country!",
          "Today marks a great milestone in healthcare reform."]
sample_tfidf = vectorizer.transform(sample)
print(model.predict(sample_tfidf))

[0 0]


In [41]:
train_texts = X_train.tolist()
test_texts = X_test.tolist()
train_labels = y_train.astype(int).tolist()
test_labels = y_test.astype(int).tolist()

# Build tf.data datasets (strings here are okay because we've disabled XLA; but we avoid graph-time string args)
train_ds = tf.data.Dataset.from_tensor_slices((train_texts, train_labels)).shuffle(1000).batch(16)
test_ds = tf.data.Dataset.from_tensor_slices((test_texts, test_labels)).batch(16)


In [42]:
import keras_nlp
model2 = keras_nlp.models.BertClassifier.from_preset("bert_base_en_uncased", num_classes=2)

100%|██████████| 761/761 [00:00<00:00, 1.47MB/s]


In [43]:
model2.compile(
    optimizer=tf.keras.optimizers.Adam(3e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [44]:
history = model2.fit(train_ds, validation_data=test_ds, epochs=3)

Epoch 1/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 479s 2s/step - accuracy: 0.7289 - loss: 0.5523 - val_accuracy: 0.7840 - val_loss: 0.4654
Epoch 2/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 392s 2s/step - accuracy: 0.8127 - loss: 0.3989 - val_accuracy: 0.7880 - val_loss: 0.4928
Epoch 3/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 391s 2s/step - accuracy: 0.8839 - loss: 0.2851 - val_accuracy: 0.7640 - val_loss: 0.5855


In [46]:
model2.evaluate(test_ds)

63/63 ━━━━━━━━━━━━━━━━━━━━ 33s 510ms/step - accuracy: 0.7801 - loss: 0.5465


[0.5854712128639221, 0.7639999985694885]

In [48]:
samples = [
    "The government's new reforms will help the economy.",
    "This corrupt administration is ruining the nation!"
]

In [51]:
preds = model2.predict(samples)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 933ms/step


In [55]:
preds

array([[ 1.0790114, -1.4878988],
       [-1.2660525,  1.2767723]], dtype=float32)

In [49]:
labels = tf.argmax(preds, axis=1).numpy()

In [56]:
labels

array([0, 1])

In [57]:
for text, label in zip(samples, labels):
    print(f"{text} -> {'partisan' if label==1 else 'neutral'}")

The government's new reforms will help the economy. -> neutral
This corrupt administration is ruining the nation! -> partisan
